## In this notebook, we test the model (which was trained on _Corpus Nummorum_ data) on three _Numismatics_ datasets (Seleucid, BIGR, and PELLA)

Plan of attack
* Downloading fine-tuned ImageNet 21k model checkpoint from .pth file
* Saving image and motif metadata for three additional datasets
* Checking the contents of the saved files
* Testing the model on the new datasets using the checkpoint .pth file
* Recording average precision (AP) for each motif and mean average precision (mAP) for each of the three datasets

Import statements and necessary functions

In [ ]:
# from google.colab import runtime
# runtime.unassign()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/coins_project/')
from utils import *

Investigating .tar file of Corpus Nummorum images

In [ ]:
tar_path = "/content/drive/MyDrive/coins_project/coin_images.tar"

with tarfile.open(tar_path, "r:*") as tar:
    tar.list()

Streaming output truncated to the last 5000 lines.
?rw------- root/root      21378 2026-06-16 10:54:31 ./wreath/CN_type_20079_BNF_41767911_cn_coin_23314_o_obv.jpg 
?rw------- root/root      15995 2026-06-16 10:54:31 ./wreath/CN_type_20079_MK_242_cn_coin_45896_p_obv.jpg 
?rw------- root/root      22313 2026-06-16 10:54:31 ./wreath/CN_type_20086_BNF_41767912_cn_coin_23315_o_obv.jpg 
?rw------- root/root      14749 2026-06-16 10:54:31 ./wreath/CN_type_20086_MK_Platzhalter_cn_coin_45919_p_obv.jpg 
?rw------- root/root      24903 2026-06-16 10:54:31 ./wreath/CN_type_20087_BNF_41767914_cn_coin_23317_o_obv.jpg 
?rw------- root/root      13771 2026-06-16 10:54:31 ./wreath/CN_type_20087_MK_41767914_cn_coin_23317_p_obv.jpg 
?rw------- root/root      16083 2026-06-16 10:54:31 ./wreath/CN_type_20087_MK_Platzhalter_cn_coin_45947_p_obv.jpg 
?rw------- root/root       9896 2026-06-16 10:54:31 ./wreath/CN_type_2009_cn_coin_9874_p_obv.jpg 
?rw------- root/root      20503 2026-06-16 10:54:31 ./wreath/CN

"Scraping" databases for coin images and metadata

In [ ]:
# set naming convention for extracted coin image datasets
dataset_str = "pella"
# dataset_str = "bigr"
# dataset_str = "seleucid"

BASE_URL = "https://numismatics.org/" + dataset_str + "/results?q=&start={}"
OUTPUT_DIR = OUTPUT_DIR = "/content/drive/MyDrive/coins_project"
IMAGE_DIR = os.path.join(OUTPUT_DIR, dataset_str + "_images")

# creates a directory in which to save images
os.makedirs(IMAGE_DIR, exist_ok=True)

# session for scraping/crawling
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})


def parse_page(start):

    url = BASE_URL.format(start)

    r = session.get(url, timeout=30)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    page_records = []

    for coin in soup.select("div.result-doc"):

        # records ID

        h4 = coin.find("h4")
        if h4 is None:
            continue

        a = h4.find("a")
        if a is None:
            continue

        record_id = a["href"].split("/")[-1]

        # gets image

        image_url = None

        thumb = coin.select_one("a.thumbImage")

        if thumb is not None:
            image_url = thumb.get("href")

        # gets metadata

        metadata = {}

        for dt, dd in zip(
            coin.find_all("dt"),
            coin.find_all("dd")
        ):
            metadata[
                dt.get_text(strip=True)
            ] = dd.get_text(" ", strip=True)

        page_records.append({

            "RecordId": record_id,

            "ImageURL": image_url,

            "Obverse": metadata.get("Obverse"),

            "Reverse": metadata.get("Reverse"),

            "Date": metadata.get("Date"),

            "Denomination": metadata.get("Denomination"),

            "Weight": metadata.get("Weight (in g)")
        })

    return page_records


# downloads image
def download_image(url, filename):

    if url is None:
        return False

    try:

        r = session.get(url, timeout=30)

        if r.status_code != 200:
            return False

        with open(filename, "wb") as f:
            f.write(r.content)

        return True

    except Exception:

        return False

records = []

TOTAL = 5000
PAGE_SIZE = 20

# crawls site
for start in tqdm(range(0, TOTAL, PAGE_SIZE)):

    page = parse_page(start)

    for coin in page:

        # downloads image
        image_name = f'{coin["RecordId"]}.jpg'

        image_path = os.path.join(
            IMAGE_DIR,
            image_name
        )

        success = download_image(
            coin["ImageURL"],
            image_path
        )

        if not success:
            image_name = None

        # stores metadata
        records.append({

            "RecordId": coin["RecordId"],

            "Image": image_name,

            "Obverse": coin["Obverse"],

            "Reverse": coin["Reverse"],

            "Date": coin["Date"],

            "Denomination": coin["Denomination"],

            "Weight": coin["Weight"]

        })

    # delay for server
    time.sleep(0.1)

# saves dataset metadata as csv
df = pd.DataFrame(records)
df.to_csv(
    os.path.join(OUTPUT_DIR, dataset_str + "_coins.csv"),
    index=False
)

print(df.head())
print()
print("Total records:", len(df))

100%|██████████| 250/250 [50:41<00:00, 12.17s/it]


            RecordId                  Image  \
0  pella.philip_ii.1                   None   
1  pella.philip_ii.2  pella.philip_ii.2.jpg   
2  pella.philip_ii.3                   None   
3  pella.philip_ii.4  pella.philip_ii.4.jpg   
4  pella.philip_ii.5  pella.philip_ii.5.jpg   

                                             Obverse  \
0                     Laureate head of Zeus to right   
1   Head of youth to right, hair tied with strophion   
2                     Laureate head of Zeus to right   
3                     Laureate head of Zeus to right   
4  Head of beardless Heracles right wearing lion ...   

                                             Reverse               Date  \
0  ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...  359 BCE - 348 BCE   
1  ΦΙΛΙΠΠΟΥ: Nude youth riding horse to left, rei...  359 BCE - 348 BCE   
2  ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...  359 BCE - 348 BCE   
3  ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...  359 BCE - 348 BCE   
4  ΦΙ

Investigates the .tar paths of coin datasets once they are saved

In [ ]:
src = Path("/content/drive/MyDrive/coins_project/" + dataset_str + "_images")  # Drive folder containing motif/image folders
tar_path = Path("/content/drive/MyDrive/coins_project/" + dataset_str + "_images.tar")

print("source:", src)
print("tar:", tar_path)

!tar -czf "$tar_path" -C "$src" .

source: /content/drive/MyDrive/coins_project/pella_images
tar: /content/drive/MyDrive/coins_project/pella_images.tar
^C


Moving on to booting up the deep learning model using a .pth file and testing it on new datasets

In [ ]:
checkpoint_path = "/content/drive/MyDrive/coins_project/coin_motif_vit_checkpoint.pth"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [ ]:
num_motifs = 7

model = timm.create_model(
    "vit_base_patch16_224.augreg_in21k",
    pretrained=False,
    num_classes=num_motifs,
)

model.to(device)

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

In [ ]:
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)

In [ ]:
print(checkpoint.keys())

dict_keys(['model_name', 'model_state_dict', 'sample_motifs', 'thresholds', 'best_score'])


In [ ]:
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [ ]:
# evaluates model
model.eval()

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

In [ ]:
config = resolve_data_config({}, model=model)

transform = create_transform(
    **config,
    is_training=False,
)

Test testing procedure on a random image

In [ ]:
# image = Image.open("/content/drive/MyDrive/coins_project/CoinsDataset/CN_dataset_nlp_objects/bull/CN_type_1417_cn_coin_2220_p_rev.jpg").convert("RGB")
image = Image.open("/content/drive/MyDrive/coins_project/pella_images/pella.philip_ii.10.jpg").convert("RGB")

x = transform(image).unsqueeze(0).to(device)

In [ ]:
with torch.no_grad():
    logits = model(x)

probabilities = torch.softmax(logits, dim=1)

confidence, prediction = probabilities.max(dim=1)

print(prediction.item())
print(confidence.item())

6
0.999992847442627


Looks good!

Lastly, test the model on the other coin datasets

In [ ]:
base_path = Path('/content/drive/MyDrive/coins_project')

# pella_copy_path = base_path / 'pella_coins_copy.csv'
pella_csv_path = base_path / 'pella_coins.csv'
pella_images_path = base_path / 'pella_images'
bigr_csv_path = base_path / 'bigr_coins.csv'
bigr_images_path = base_path / 'bigr_images'
seleucid_csv_path = base_path / 'seleucid_coins.csv'
seleucid_images_path = base_path / 'seleucid_images'
cn_nlp_path = base_path / 'CoinsDataset' / 'CN_dataset_nlp_objects'

if base_path.is_dir():
  print(f"Found directory: {base_path}")
else:
  print(f"Directory not found at {base_path}")
for p in [pella_csv_path, bigr_csv_path, seleucid_csv_path]:
  if p.is_file():
    print(f"Found file: {p}")
  else:
    print(f"File not found at {p}")
for p in [pella_images_path, bigr_images_path, seleucid_images_path, cn_nlp_path]:
  if p.is_dir():
    print(f"Found directory: {p}")
  else:
    print(f"Directory not found at {p}")

Found directory: /content/drive/MyDrive/coins_project
Found file: /content/drive/MyDrive/coins_project/pella_coins_copy.csv
Found file: /content/drive/MyDrive/coins_project/pella_coins.csv
Found file: /content/drive/MyDrive/coins_project/bigr_coins.csv
Found directory: /content/drive/MyDrive/coins_project/pella_images
Found directory: /content/drive/MyDrive/coins_project/bigr_images
Found directory: /content/drive/MyDrive/coins_project/seleucid_images
Found directory: /content/drive/MyDrive/coins_project/CoinsDataset/CN_dataset_nlp_objects


In [ ]:
# collects all the motifs in the CN dataset and how often they appear.
# takes a few minutes to run
motifs_with_counts = {}
for motif in os.listdir(cn_nlp_path):
  p = cn_nlp_path / motif
  if p.is_dir():
    motifs_with_counts[motif] = len(os.listdir(cn_nlp_path / motif))

print(motifs_with_counts)

{'abacus': 5, 'abundantia': 3, 'acrostolium': 11, 'acroteria': 1, 'actaeon': 3, 'aegis': 432, 'aeneas': 57, 'aequitas': 180, 'agonistic_crown': 2, 'agrippa': 7, 'agrippina_minor': 2, 'alexander_iii': 145, 'altar': 1049, 'amphora': 247, 'anchialos': 2, 'anchises': 60, 'anchor': 32, 'androclus': 3, 'andromeda': 3, 'animal': 13, 'annona': 2, 'antinous': 5, 'antiochus_ii_theos': 2, 'antlers': 24, 'antonia_minor': 6, 'antoninus_pius': 892, 'anubis': 15, 'aphrodite': 65, 'apis': 35, 'aplustre': 84, 'apollo': 2774, 'apollon': 9, 'apple': 201, 'arch': 12, 'archer': 2, 'ares': 224, 'ariadne': 7, 'arm': 1907, 'armour': 38, 'arrow': 490, 'artemis': 923, 'ascanius': 57, 'asclepius': 1029, 'astragal': 6, 'athena': 4108, 'athlete': 115, 'attis': 7, 'augustus': 384, 'aulos': 1, 'bag': 3, 'barley': 49, 'base': 268, 'basin': 14, 'basket': 90, 'bear': 19, 'bee': 29, 'beehive': 6, 'belt': 4, 'berry': 82, 'biga': 125, 'bird': 49, 'boar': 144, 'board': 65, 'bonus_eventus': 102, 'boot': 138, 'bow': 1509, 'b

In [ ]:
# Reduce to the most frequently appearing motifs.
# This seems to be about the cutoff in the model-training/testing file.
# That is, it's about 0.1% of the length of the CN dataset.
MIN_MOTIF_FREQUENCY = 150
frequent_motifs = {motif: count for (motif,count) in motifs_with_counts.items() if count >= MIN_MOTIF_FREQUENCY}
print(frequent_motifs)

{'aegis': 432, 'aequitas': 180, 'altar': 1049, 'amphora': 247, 'antoninus_pius': 892, 'apollo': 2774, 'apple': 201, 'ares': 224, 'arm': 1907, 'arrow': 490, 'artemis': 923, 'asclepius': 1029, 'athena': 4108, 'augustus': 384, 'base': 268, 'bow': 1509, 'branch': 939, 'bull': 1342, 'bust': 9423, 'caduceus': 596, 'cantharus': 617, 'cap': 191, 'caracalla': 2238, 'chlamys': 352, 'cista': 507, 'club': 1014, 'column': 255, 'commodus': 947, 'corn': 1027, 'cornucopia': 1703, 'corn_wreath': 530, 'crepidoma': 373, 'crescent': 261, 'crispina': 159, 'cuirass': 5257, 'cybele': 318, 'demeter': 971, 'diadem': 1455, 'diadumenian': 161, 'dionysus': 1455, 'dolphin': 1198, 'domitian': 209, 'double_ax': 154, 'double_chiton': 182, 'eagle': 1564, 'ear': 694, 'earring': 632, 'elagabalus': 798, 'emperor': 636, 'faustina_minor': 391, 'figure': 172, 'foot': 1901, 'gallienus': 154, 'garment': 292, 'geta': 682, 'gordian': 735, 'gorgoneion': 450, 'grain': 485, 'grape': 1165, 'griffin': 1152, 'hadrian': 296, 'hair': 7

In [ ]:
# Let's see what we're working with...
df = open_numismatics_data(pella_csv_path, pella_images_path)
df.head(10)

,RecordId,filename,Obverse,Reverse,Date,Denomination,image_path
1,pella.philip_ii.2,pella.philip_ii.2.jpg,"Head of youth to right, hair tied with strophion","ΦΙΛΙΠΠΟΥ: Nude youth riding horse to left, rei...",359 BCE - 348 BCE,Hemidrachm,/content/drive/MyDrive/coins_project/pella_ima...
3,pella.philip_ii.4,pella.philip_ii.4.jpg,Laureate head of Zeus to right,"ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...",359 BCE - 348 BCE,Tetradrachm,/content/drive/MyDrive/coins_project/pella_ima...
4,pella.philip_ii.5,pella.philip_ii.5.jpg,Head of beardless Heracles right wearing lion ...,"ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...",359 BCE - 348 BCE,Didrachm,/content/drive/MyDrive/coins_project/pella_ima...
7,pella.philip_ii.8,pella.philip_ii.8.jpg,Laureate head of Zeus to right,"ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...",359 BCE - 348 BCE,Tetradrachm,/content/drive/MyDrive/coins_project/pella_ima...
9,pella.philip_ii.10,pella.philip_ii.10.jpg,Head of beardless Heracles right wearing lion ...,"ΦΙΛΙΠΠΟΥ: Nude youth riding horse to left, rei...",359 BCE - 348 BCE,Drachma,/content/drive/MyDrive/coins_project/pella_ima...
10,pella.philip_ii.11,pella.philip_ii.11.jpg,Laureate head of Zeus to right,"ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...",359 BCE - 348 BCE,Tetradrachm,/content/drive/MyDrive/coins_project/pella_ima...
11,pella.philip_ii.12,pella.philip_ii.12.jpg,Laureate head of Zeus to right,"ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...",359 BCE - 348 BCE,Tetradrachm,/content/drive/MyDrive/coins_project/pella_ima...
12,pella.philip_ii.13,pella.philip_ii.13.jpg,Head of beardless Heracles right wearing lion ...,"ΦΙΛΙΠΠΟΥ: Nude youth riding horse to left, rei...",359 BCE - 348 BCE,Drachma,/content/drive/MyDrive/coins_project/pella_ima...
13,pella.philip_ii.14,pella.philip_ii.14.jpg,"Head of youth to right, hair tied with strophion","ΦΙΛΙΠΠΟΥ: Nude youth riding horse to left, rei...",359 BCE - 348 BCE,Hemidrachm,/content/drive/MyDrive/coins_project/pella_ima...
17,pella.philip_ii.18,pella.philip_ii.18.jpg,Laureate head of Zeus to right,"ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...",359 BCE - 348 BCE,Tetradrachm,/content/drive/MyDrive/coins_project/pella_ima...


In [ ]:
df = check_for_motifs(df, frequent_motifs.keys(), "Obverse")
df.head(20)

,RecordId,filename,Obverse,Reverse,Date,Denomination,image_path,aegis,aequitas,altar,...,tunny,tyche,urn,veil,vine,wheel,wing,woman,wreath,zeus
1,pella.philip_ii.2,pella.philip_ii.2.jpg,"Head of youth to right, hair tied with strophion","ΦΙΛΙΠΠΟΥ: Nude youth riding horse to left, rei...",359 BCE - 348 BCE,Hemidrachm,/content/drive/MyDrive/coins_project/pella_ima...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,pella.philip_ii.4,pella.philip_ii.4.jpg,Laureate head of Zeus to right,"ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...",359 BCE - 348 BCE,Tetradrachm,/content/drive/MyDrive/coins_project/pella_ima...,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,pella.philip_ii.5,pella.philip_ii.5.jpg,Head of beardless Heracles right wearing lion ...,"ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...",359 BCE - 348 BCE,Didrachm,/content/drive/MyDrive/coins_project/pella_ima...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,pella.philip_ii.8,pella.philip_ii.8.jpg,Laureate head of Zeus to right,"ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...",359 BCE - 348 BCE,Tetradrachm,/content/drive/MyDrive/coins_project/pella_ima...,0,0,0,...,0,0,0,0,0,0,0,0,0,1
9,pella.philip_ii.10,pella.philip_ii.10.jpg,Head of beardless Heracles right wearing lion ...,"ΦΙΛΙΠΠΟΥ: Nude youth riding horse to left, rei...",359 BCE - 348 BCE,Drachma,/content/drive/MyDrive/coins_project/pella_ima...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10,pella.philip_ii.11,pella.philip_ii.11.jpg,Laureate head of Zeus to right,"ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...",359 BCE - 348 BCE,Tetradrachm,/content/drive/MyDrive/coins_project/pella_ima...,0,0,0,...,0,0,0,0,0,0,0,0,0,1
11,pella.philip_ii.12,pella.philip_ii.12.jpg,Laureate head of Zeus to right,"ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...",359 BCE - 348 BCE,Tetradrachm,/content/drive/MyDrive/coins_project/pella_ima...,0,0,0,...,0,0,0,0,0,0,0,0,0,1
12,pella.philip_ii.13,pella.philip_ii.13.jpg,Head of beardless Heracles right wearing lion ...,"ΦΙΛΙΠΠΟΥ: Nude youth riding horse to left, rei...",359 BCE - 348 BCE,Drachma,/content/drive/MyDrive/coins_project/pella_ima...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
13,pella.philip_ii.14,pella.philip_ii.14.jpg,"Head of youth to right, hair tied with strophion","ΦΙΛΙΠΠΟΥ: Nude youth riding horse to left, rei...",359 BCE - 348 BCE,Hemidrachm,/content/drive/MyDrive/coins_project/pella_ima...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
17,pella.philip_ii.18,pella.philip_ii.18.jpg,Laureate head of Zeus to right,"ΦΙΛΙΠΠΟΥ: Philip II wearing causia, tunic, and...",359 BCE - 348 BCE,Tetradrachm,/content/drive/MyDrive/coins_project/pella_ima...,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [ ]:
motif_counts = df[frequent_motifs.keys()].sum()
motif_counts[motif_counts >= 5]

,0
apollo,74
athena,459
bow,65
diadem,23
ear,3015
griffin,48
hair,48
head,3261
helmet,459
hera,2522


In [ ]:
test_df = df.copy()

sample_motifs = ["eagle", "throne", "snake", "bull", "horse", "star", "head"]

data_config = resolve_model_data_config(model)
eval_tfms = create_transform(**data_config, is_training=False)

test_ds = CoinMotifDataset(test_df, sample_motifs, eval_tfms)

batch_size = 32

test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

In [ ]:
test_probs, test_y = predict(
    model=model,
    loader=test_loader,
    device=device,
    desc="Test",
)

for j, motif in enumerate(sample_motifs):
    positives = int(test_y[:, j].sum())

    if positives > 0:
        ap = average_precision_score(test_y[:, j], test_probs[:, j])
        print(f"{motif}: AP={ap:.4f}, positives={positives}")
    else:
        print(f"{motif}: AP=undefined, positives=0")

Test:   0%|          | 0/104 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
                                                       

eagle: AP=undefined, positives=0
throne: AP=undefined, positives=0
snake: AP=0.0103, positives=7
bull: AP=undefined, positives=0
horse: AP=undefined, positives=0
star: AP=undefined, positives=0
head: AP=0.9999, positives=3261


Results of testing model on Corpus Nummorum Greco-Roman coins as well as Numismatics PELLA/BIGR/Seleucid coins

In [ ]:
# # CORPUS NUMMORUM (mAP = 0.824)
# eagle: AP=0.7564, positives=317
# throne: AP=0.8305, positives=281
# snake: AP=0.7304, positives=479
# bull: AP=0.8280, positives=265
# horse: AP=0.8858, positives=493
# star: AP=0.8288, positives=122
# head: AP=0.9110, positives=3612

In [ ]:
# # BIGR (mAP = 0.384)
# eagle: AP=0.0588, positives=1
# throne: AP=0.4306, positives=3
# snake: AP=undefined, positives=0
# bull: AP=0.1115, positives=49
# horse: AP=1.0000, positives=3
# star: AP=0.1102, positives=2
# head: AP=0.5916, positives=75

In [ ]:
# # PELLA (mAP = 0.505)
# eagle: AP=undefined, positives=0
# throne: AP=undefined, positives=0
# snake: AP=0.0103, positives=7
# bull: AP=undefined, positives=0
# horse: AP=undefined, positives=0
# star: AP=undefined, positives=0
# head: AP=0.9999, positives=3289

In [ ]:
# # SELEUCID (mAP = 0.386)
# eagle: AP=undefined, positives=0
# throne: AP=0.8209, positives=13
# snake: AP=undefined, positives=0
# bull: AP=0.0349, positives=9
# horse: AP=0.1120, positives=10
# star: AP=0.0107, positives=10
# head: AP=0.9533, positives=1062